# 01 — EDA: FineBadminton Dataset

Exploratory analysis of the FineBadminton annotation JSON.  
Covers: dataset structure, shot-type distribution, strategy distribution, and shot × strategy correlations.

In [ ]:
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter, defaultdict
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')

ANNOT_PATH = '../Datasets/FineBadminton-dataset/dataset/transformed_combined_rounds_output_en_evals_translated.json'

## 1. Load annotations

In [ ]:
with open(ANNOT_PATH) as f:
    data = json.load(f)

# Flatten all shots
shots = []
for rally in data:
    for hit in rally.get('hitting', []):
        hit['video'] = rally['video']
        shots.append(hit)

df = pd.DataFrame(shots)
df['has_strategy'] = df['strategies'].apply(lambda x: isinstance(x, list) and len(x) > 0)

print(f'Rallies:              {len(data)}')
print(f'Total shots:          {len(df)}')
print(f'Shots with strategy:  {df["has_strategy"].sum()}')
print(f'Unique shot types:    {df["hit_type"].nunique()}')
print(f'Unique players:       {df["player"].nunique()}')
df.head(3)

## 2. Shot-type distribution

In [ ]:
shot_counts = df['hit_type'].value_counts().drop('', errors='ignore')

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(shot_counts.index, shot_counts.values, color=sns.color_palette('muted', len(shot_counts)))
ax.bar_label(bars, padding=3)
ax.set_title('Shot-type distribution (FineBadminton, n=414)', fontsize=13)
ax.set_ylabel('Count')
ax.set_xlabel('Shot type')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../results/eda_shot_distribution.png', dpi=150)
plt.show()

## 3. Strategy distribution

In [ ]:
all_strats = [s for strats in df['strategies'].dropna() for s in (strats if isinstance(strats, list) else [])]
strat_counts = Counter(all_strats)

labels, values = zip(*strat_counts.most_common())
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(labels, values, color=sns.color_palette('Set2', len(labels)))
ax.bar_label(bars, padding=3)
ax.set_title('Strategy label distribution (FineBadminton)', fontsize=13)
ax.set_ylabel('Count')
ax.set_xlabel('Strategy')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../results/eda_strategy_distribution.png', dpi=150)
plt.show()

## 4. Shot × Strategy co-occurrence heatmap

In [ ]:
# Build raw count matrix
cross = defaultdict(Counter)
for _, row in df.iterrows():
    ht = row['hit_type']
    if not ht or not isinstance(row.get('strategies'), list):
        continue
    for st in row['strategies']:
        cross[ht][st] += 1

shot_types_ordered = [t for t, _ in Counter(df['hit_type']).most_common() if t]
strat_ordered = [s for s, _ in strat_counts.most_common()]

matrix = pd.DataFrame(
    [[cross[ht][st] for st in strat_ordered] for ht in shot_types_ordered],
    index=shot_types_ordered, columns=strat_ordered
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(matrix, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('Shot type × Strategy co-occurrence (raw counts)', fontsize=13)
ax.set_xlabel('Strategy')
ax.set_ylabel('Shot type')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../results/eda_shot_strategy_heatmap.png', dpi=150)
plt.show()

## 5. Conditional probability P(strategy | shot_type)

In [ ]:
# Row-normalise: P(strategy | shot_type)
row_sums = matrix.sum(axis=1).replace(0, np.nan)
prob_matrix = matrix.div(row_sums, axis=0)

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(prob_matrix, annot=True, fmt='.0%', cmap='Blues', linewidths=0.5, ax=ax,
            vmin=0, vmax=1)
ax.set_title('P(strategy | shot_type) — row-normalised', fontsize=13)
ax.set_xlabel('Strategy')
ax.set_ylabel('Shot type')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../results/eda_shot_strategy_prob.png', dpi=150)
plt.show()

## 6. Strategy entropy per shot type

Higher entropy = the shot type is used across many different strategies (harder to classify).  
Lower entropy = one dominant strategy (near-deterministic, easier).

In [ ]:
def entropy(row):
    total = row.sum()
    if total == 0:
        return 0.0
    probs = row[row > 0] / total
    return -(probs * np.log2(probs)).sum()

entropies = matrix.apply(entropy, axis=1).sort_values()

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#d73027' if e > 2.0 else '#4575b4' for e in entropies]
bars = ax.barh(entropies.index, entropies.values, color=colors)
ax.axvline(x=2.0, color='grey', linestyle='--', linewidth=1, label='H=2 threshold')
ax.set_title('Strategy entropy per shot type (higher = more ambiguous)', fontsize=12)
ax.set_xlabel('Shannon entropy (bits)')
ax.legend()
plt.tight_layout()
plt.savefig('../results/eda_strategy_entropy.png', dpi=150)
plt.show()

print('Near-deterministic shots (H < 1.5):')
print(entropies[entropies < 1.5])
print('\nAmbiguous shots (H > 2.0):')
print(entropies[entropies > 2.0])

## 7. Dominant strategy per shot type (summary table)

In [ ]:
rows = []
shot_totals = defaultdict(int)
for _, row in df.iterrows():
    ht = row['hit_type']
    if ht and isinstance(row.get('strategies'), list) and row['strategies']:
        shot_totals[ht] += 1

for ht in shot_types_ordered:
    if ht not in cross:
        continue
    top_strat, top_count = cross[ht].most_common(1)[0]
    total = shot_totals[ht]
    n_strats = len(cross[ht])
    H = entropy(matrix.loc[ht])
    rows.append({
        'Shot type': ht,
        'n (with strategy)': total,
        'Top strategy': top_strat,
        'P(top | shot)': f'{top_count/total:.0%}',
        'Num strategies': n_strats,
        'Entropy (bits)': f'{H:.2f}',
    })

summary = pd.DataFrame(rows).set_index('Shot type')
display(summary)

## 8. Key findings

### Near-deterministic shot → strategy mappings
| Shot type | Dominant strategy | P(strategy\|shot) |
|---|---|---|
| kill | intercept | **99%** |
| net kill | seamlessly | **100%** |
| drop shot | intercept | **96%** |
| net lift | To create depth | **88%** |
| block | defensive / passive | **79%** |

Kill, drop shot, and net kill have near-perfect strategy predictability from the shot label alone — skeleton features may add marginal value for these classes.

### Ambiguous shots (where pose context matters most)
| Shot type | Entropy | Why ambiguous |
|---|---|---|
| net shot | 2.69 bits | Used across 7 strategies in both attack/defense |
| push shot | 2.56 bits | Transitional shot — context-dependent |
| drive | 2.31 bits | Intercept and passive co-occur |
| cross-court net shot | 2.38 bits | Defensive or attacking depending on court position |

These are where **skeleton + court position features should contribute the most discriminative power**.

### Strategy → shot diversity
- **intercept** is concentrated in kill/drop/drive — clean signal
- **To create depth** is almost exclusively push shot + net lift (2 shot types)
- **passive** / **defensive** span 7–9 shot types — these are the hardest strategy labels to model

### Class imbalance considerations
- Shot level: push shot (84) and kill (70) dominate; net kill (8) and drop shot (23) are rare
- Strategy level: passive (137) and intercept (120) are 2× more frequent than deception (21)
- Episodic sampling in ProtoNet training directly addresses this imbalance

## 9. Rally-level statistics

In [ ]:
rally_lengths = [len(r.get('hitting', [])) for r in data]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(rally_lengths, bins=15, color='steelblue', edgecolor='white')
axes[0].set_title('Shots per rally distribution')
axes[0].set_xlabel('Number of shots')
axes[0].set_ylabel('Frequency')

# Shots per video (proxy for rally difficulty)
shots_per_video = df.groupby('video').size().sort_values(ascending=False)
axes[1].bar(range(len(shots_per_video)), shots_per_video.values, color='coral')
axes[1].set_title('Shots per rally (video)')
axes[1].set_xlabel('Rally index (sorted)')
axes[1].set_ylabel('Shot count')

plt.tight_layout()
plt.savefig('../results/eda_rally_stats.png', dpi=150)
plt.show()

print(f'Mean shots/rally:   {np.mean(rally_lengths):.1f}')
print(f'Median shots/rally: {np.median(rally_lengths):.0f}')
print(f'Max shots/rally:    {max(rally_lengths)}')
print(f'Min shots/rally:    {min(rally_lengths)}')